# E16 — 이탈·복귀·종료 진단 (재학습 없음)
첨부 envs.py에 잔디 조기 종료 규칙은 없습니다. 정책·보상·종료 조건을 유지하고 원본 프레임마다 진단만 기록합니다. 내부 상태를 행동에 전달하지 않습니다. 접선 기반 방향 판정은 근사이며 영상과 대조하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, shutil, tempfile
ROOT=Path('/content/drive/MyDrive/261103')
RESULTS=ROOT/'261103_results'
if not (ROOT/'train.py').exists():
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(['git','clone','--depth','1','--branch','261103','https://github.com/MilkySon/RL_carracing_competition.git',tmp+'/repo'],check=True)
        shutil.copytree(Path(tmp)/'repo'/'261103',ROOT,dirs_exist_ok=True)
RESULTS.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
os.environ['SDL_VIDEODRIVER']='dummy'
os.environ['SDL_AUDIODRIVER']='dummy'
print(ROOT,RESULTS)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','swig'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import torch
print('GPU:',torch.cuda.is_available())
def run(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a') as f:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='');f.write(line);f.flush()
        rc=p.wait()
    if rc:raise RuntimeError(f'Failed ({rc}): {log}')


## 1. 모델 경로
E15 먼저 실행. 기존 모델 경로는 실제 Drive 위치로 수정. 없는 이전 모델은 표시 후 건너뜁니다.

In [ ]:
CHECKPOINTS={'E15':RESULTS/'E15'/'best.pt'}
for name in ['E10','E12','E14']:
    CHECKPOINTS[name]=Path('/content/drive/MyDrive/261101carracing_results')/name/'best.pt'
OUT=RESULTS/'E16';OUT.mkdir(parents=True,exist_ok=True)
assert CHECKPOINTS['E15'].exists(),'E15 완료 또는 경로 수정 필요'


## 2. 종료 조건 및 프레임 로그
이탈=모든 바퀴의 도로 접촉 없음. 도로 재접촉과 정방향·새 타일 방문 복귀를 구별합니다. 연속 역주행 집계는 원본 프레임 단위입니다.

In [ ]:
for name,ckpt in CHECKPOINTS.items():
    if not ckpt.exists():print('SKIP missing:',name,ckpt);continue
    run([sys.executable,'audit_termination.py','--ckpt',ckpt,'--seeds','1000-1009','--out',OUT/name],OUT/f'{name}_console.log')
import pandas as pd
for p in sorted(OUT.glob('*/episodes.csv')):
    print(p.parent.name);display(pd.read_csv(p))


## 3. seed 1001 실패 시작점 영상

In [ ]:
for name,ckpt in CHECKPOINTS.items():
    if ckpt.exists():run([sys.executable,'evaluate.py','--ckpt',ckpt,'--seeds','1001','--video',OUT/name/'videos'],OUT/f'{name}_video_console.log')


## 판단 메모
frames.csv: 이탈 → 재접촉 → 정방향/새 타일 진행의 순서 확인. 종료 원인은 관측에 따른 분류이며 영상·원본 환경 코드와 대조.
즉시 종료 규칙이 없으므로 잔디 20스텝 허용을 무조건 추가하지 않습니다. 복귀 실패가 반복될 때 E17, 이후 필요 시 E18 검토.